# 43. CD-OPE-S Red Counterfactual 분석

CD-OPE-S가 red original과 grayscale/gray-world 사이의 gap을 줄였는지 확인합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch4_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/4장/ch4_utils.py")) + list(Path.cwd().glob("**/ch4_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "4장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch4_utils import *

paths = find_ch4_paths()
set_korean_font()
set_seed(41)
paths

Chapter4Paths(chapter4_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장'), chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/manifests'), design_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/chapter4_cd_ope_s_architecture_design.md'), validity_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/chapter4_cd_ope_s_design_validity_review.md'))

## 43-1. 평가할 run 선택

In [2]:
manifests = create_ch4_manifests(max_per_cell=3, seed=41)
seed_metrics = collect_ch4_cd_ope_metrics()
if seed_metrics.empty:
    raise FileNotFoundError("42번에서 CD-OPE-S 학습 run을 먼저 생성하세요.")

TARGET_VARIANT = "gl_cd_consistency_style"
TARGET_SEED = 0
selected = seed_metrics[
    (seed_metrics["variant"] == TARGET_VARIANT)
    & (seed_metrics["model_seed"] == TARGET_SEED)
]
if selected.empty:
    raise FileNotFoundError(f"선택한 run이 없습니다: {TARGET_VARIANT} seed {TARGET_SEED}")
run_dir = Path(selected.iloc[0]["run_dir"])
run_dir

WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/cd_ope_s/runs/gl_cd_consistency_style/seed_0')

## 43-2. Counterfactual 평가 실행

In [3]:
out_dir = paths.runs_root / "cd_ope_s" / "counterfactual" / TARGET_VARIANT / f"seed_{TARGET_SEED}"
metrics = evaluate_cd_ope_s_counterfactual(
    run_dir,
    manifests["eval_color_counterfactual_probe"],
    out_dir,
    transforms=DEFAULT_COUNTERFACTUALS,
    max_samples=120,
    seed=41,
)
display(metrics.head())
display(
    metrics.groupby(["transform", "color_group"])[
        ["target_dice", "target_fnr", "prediction_flip_rate"]
    ]
    .mean()
    .reset_index()
)

C:\Users\준승\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,sample_id,transform,color_group,shape_group,defect_type,target_dice,target_iou,target_fnr,target_fpr,target_margin,entropy_mean,prediction_flip_rate
0,eval_matched_matched_control_blue_dent_bottom_001,original,blue,bottom_half_metal,dent,0.873980,0.776167,0.078394,0.006179,2.880138,0.068081,0.000000
1,eval_matched_matched_control_blue_dent_bottom_001,grayscale,blue,bottom_half_metal,dent,0.866038,0.763727,0.122371,0.004918,2.545460,0.060452,0.003784
2,eval_matched_matched_control_blue_dent_bottom_001,gray_world,blue,bottom_half_metal,dent,0.863680,0.760067,0.133843,0.004602,2.531974,0.063796,0.003601
3,eval_matched_matched_control_blue_dent_bottom_001,red_to_neutral,blue,bottom_half_metal,dent,0.852368,0.742718,0.122371,0.005990,2.579463,0.074221,0.003113
4,eval_matched_matched_control_blue_dent_bottom_001,hue_rotate_90,blue,bottom_half_metal,dent,0.880074,0.785832,0.087954,0.005296,2.761447,0.064304,0.006165


,transform,color_group,target_dice,target_fnr,prediction_flip_rate
0,brightness_0p75,blue,0.570993,0.507019,0.001686
1,brightness_0p75,neutral,0.601327,0.459364,0.001361
2,brightness_0p75,purple,0.509807,0.559833,0.002136
3,brightness_0p75,red,0.396978,0.620261,0.002724
4,brightness_1p25,blue,0.563552,0.514718,0.002942
5,brightness_1p25,neutral,0.589049,0.461949,0.002202
6,brightness_1p25,purple,0.413322,0.638166,0.006485
7,brightness_1p25,red,0.399040,0.644825,0.005094
8,channel_shuffle_bgr,blue,0.595674,0.481637,0.003273
9,channel_shuffle_bgr,neutral,0.585012,0.469320,0.001348


## 43-3. red gap 계산

In [4]:
red = metrics[metrics["color_group"] == "red"]
pivot = red.groupby("transform")["target_dice"].mean()
gaps = {
    "original_vs_grayscale": float(abs(pivot.get("original", np.nan) - pivot.get("grayscale", np.nan))),
    "original_vs_gray_world": float(abs(pivot.get("original", np.nan) - pivot.get("gray_world", np.nan))),
}
gaps

{'original_vs_grayscale': 0.019968561761335768,
 'original_vs_gray_world': 0.0017215550313445238}